In [22]:
import numpy as np
import os

# Define la ruta a uno de tus archivos
archivo_ejemplo = '../Dataset/sequences/aplausos_20260525_160032_061.npy'

# Cargar el archivo
data = np.load(archivo_ejemplo)

# Imprimir la forma (shape)
print(f"Shape total: {data.shape}")

# Explicación del shape (asumiendo formato [frames, nodos, dims])
frames = data.shape[0]
nodos = data.shape[1]
dims = data.shape[2]

print("-" * 30)
print(f"Frames:  {frames}")
print(f"Nodos:   {nodos}")
print(f"Dims:    {dims}")

Shape total: (120, 79, 3)
------------------------------
Frames:  120
Nodos:   79
Dims:    3


In [2]:
data

array([[[-0.0391266 , -0.1214872 , -0.24015987],
        [-0.02738317, -0.14673206, -0.23469396],
        [-0.01800669, -0.1484316 , -0.23482405],
        ...,
        [-0.00352374, -0.16488202,  0.02795762],
        [-0.0100312 , -0.1671513 ,  0.02517943],
        [-0.01790575, -0.16614093,  0.02295721]],

       [[-0.03914313, -0.12150939, -0.23621985],
        [-0.02741231, -0.1467512 , -0.23055245],
        [-0.01803259, -0.14844526, -0.23066165],
        ...,
        [-0.0032308 , -0.16502298,  0.03048391],
        [-0.00967322, -0.1674337 ,  0.02766301],
        [-0.01754656, -0.16658458,  0.02540364]],

       [[-0.03916909, -0.1211703 , -0.23889919],
        [-0.02744831, -0.1464289 , -0.23348308],
        [-0.01805616, -0.14812656, -0.23359291],
        ...,
        [-0.00356595, -0.16455731,  0.02697469],
        [-0.01002555, -0.16700168,  0.02414651],
        [-0.01787155, -0.16616924,  0.02187042]],

       ...,

       [[-0.03920409, -0.11734895, -0.2314671 ],
        [-0

In [25]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import Image, display

# 1. Cargar tus datos 
data = np.load('../Dataset/sequences/aplausos_20260525_160032_061.npy') 
print(f"Dimensiones del dataset: {data.shape}")

# --- DEFINIR LAS CONEXIONES DEL ESQUELETO ---
CONEXIONES = [
    # --- Pose (0 a 24) ---
    (11, 12), (11, 23), (12, 24), (23, 24),
    (11, 13), (13, 15), (12, 14), (14, 16),
    (0, 1), (1, 2), (2, 3), (0, 4), (4, 5), (5, 6),
]

# [NUEVO] Conectar las muñecas (Pose) a la base de las manos (0_izq y 0_der)
# El nodo 15 es la muñeca izquierda (Pose), el nodo 25 es la base de la mano izquierda.
# El nodo 16 es la muñeca derecha (Pose), el nodo 46 es la base de la mano derecha.
CONEXIONES += [
    (15, 25), 
    (16, 46)
]

o_izq = 25
CONEXIONES += [
    (o_izq, o_izq+1), (o_izq+1, o_izq+2), (o_izq+2, o_izq+3), (o_izq+3, o_izq+4),       
    (o_izq, o_izq+5), (o_izq+5, o_izq+6), (o_izq+6, o_izq+7), (o_izq+7, o_izq+8),       
    (o_izq, o_izq+9), (o_izq+9, o_izq+10), (o_izq+10, o_izq+11), (o_izq+11, o_izq+12),  
    (o_izq, o_izq+13), (o_izq+13, o_izq+14), (o_izq+14, o_izq+15), (o_izq+15, o_izq+16),
    (o_izq, o_izq+17), (o_izq+17, o_izq+18), (o_izq+18, o_izq+19), (o_izq+19, o_izq+20) 
]

o_der = 46
CONEXIONES += [
    (o_der, o_der+1), (o_der+1, o_der+2), (o_der+2, o_der+3), (o_der+3, o_der+4),       
    (o_der, o_der+5), (o_der+5, o_der+6), (o_der+6, o_der+7), (o_der+7, o_der+8),       
    (o_der, o_der+9), (o_der+9, o_der+10), (o_der+10, o_der+11), (o_der+11, o_der+12),  
    (o_der, o_der+13), (o_der+13, o_der+14), (o_der+14, o_der+15), (o_der+15, o_der+16),
    (o_der, o_der+17), (o_der+17, o_der+18), (o_der+18, o_der+19), (o_der+19, o_der+20) 
]

c_izq = 67
CONEXIONES += [(c_izq, c_izq+1), (c_izq+1, c_izq+2), (c_izq+2, c_izq+3)]

c_der = 71
CONEXIONES += [(c_der, c_der+1), (c_der+1, c_der+2), (c_der+2, c_der+3)]

CONEXIONES += [
    (77, 75), 
    (75, 78), 
    (78, 76), 
    (76, 77), 
    (75, 76)  
]

CONEXIONES = [(i, j) for i, j in CONEXIONES if i < 79 and j < 79]

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

# [NUEVO] Ajuste de la perspectiva de la cámara
# view_init controla desde dónde miras el gráfico.
# elev=-90 (mirar desde abajo hacia arriba o ajustar según necesidad, invertimos ejes en su lugar)
# azim=-90 (gira la vista para estar de frente al eje XY)
ax.view_init(elev=-90, azim=-90)

x_min, x_max = data[:, :, 0].min(), data[:, :, 0].max()
y_min, y_max = data[:, :, 1].min(), data[:, :, 1].max()
# [NUEVO] El eje Z en MediaPipe suele tener una escala mucho menor.
# Si lo dejas libre, matplotlib lo estira y deforma la figura.
# Fijamos un rango Z arbitrario pero proporcionado a X e Y para evitar distorsión.
rango_max = max(x_max - x_min, y_max - y_min) / 2.0
z_mid = (data[:, :, 2].max() + data[:, :, 2].min()) / 2.0

margin = 0.1
ax.set_xlim([x_min - margin, x_max + margin])
ax.set_ylim([y_min - margin, y_max + margin])
# [NUEVO] Forzar que el eje Z (profundidad) tenga una proporción similar
ax.set_zlim([z_mid - rango_max, z_mid + rango_max])

# [NUEVO] Solo invertimos X si notas que tu mano derecha sale a la izquierda (efecto espejo de cv2.flip)
# ax.invert_xaxis() 
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

scatter = ax.scatter([], [], [], c='cyan', marker='o', edgecolors='blue', s=20)
lineas = [ax.plot([], [], [], color='red', linewidth=1.5)[0] for _ in CONEXIONES]

def update(frame):
    ax.set_title(f"Secuencia (79 Nodos) - Frame {frame + 1} / {data.shape[0]}")
    
    x = data[frame, :, 0]
    y = data[frame, :, 1]
    z = data[frame, :, 2]
    
    scatter._offsets3d = (x, y, z)
    
    for linea, (i, j) in zip(lineas, CONEXIONES):
        linea.set_data([x[i], x[j]], [y[i], y[j]])
        linea.set_3d_properties([z[i], z[j]])
        
    return [scatter] + lineas

fps_deseado = 30
ani = FuncAnimation(fig, update, frames=data.shape[0], interval=1000/fps_deseado, blit=False)

nombre_gif = 'secuencia_nueva_79nodos.gif'
ani.save(nombre_gif, writer='pillow', fps=fps_deseado)

plt.close()
print(f"¡Archivo '{nombre_gif}' guardado con éxito!")

#display(Image(filename=nombre_gif))

Dimensiones del dataset: (120, 79, 3)
¡Archivo 'secuencia_nueva_79nodos.gif' guardado con éxito!
